# Chapter 6 &mdash; Merging Equivalence Classes into the Minimal Machine

**Concept 9 of the Chapter 6 decomposition:** *Merging Equivalence Classes into the Minimal Machine*

Pairs still at $-1$ are equivalent; overlapping pairs coalesce into classes, one class per minimal state.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Merging-Equivalence-Classes/Concept-Merging-Equivalence-Classes.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The frame table leaves a set of equivalent **pairs**. Those pairs **overlap**: if
$(p,q)$ and $(q,r)$ are both equivalent, then $p$, $q$ and $r$ all belong to one
class &mdash; indistinguishability is transitive.

**Bashing** the pairs together (Jove's `bash_eql_classes`) coalesces them into
equivalence classes. Each class becomes **one state** of the minimal DFA; a
representative is chosen (`mk_rep_eqc`), transitions are lifted class-wise, and the
class containing $q_0$ is the new start state.

Only reachable states may be merged this way, which is why pruning comes first.

## 2. Definitions

### A machine with a three-way merge

In [ ]:
D = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 -> F1
A  : 1 -> F2
B  : 0 -> F2
B  : 1 -> F3
F1 : 0 | 1 -> F1
F2 : 0 | 1 -> F2
F3 : 0 | 1 -> F3
''')

### Equivalence classes, by transitive closure of the equivalent pairs

In [ ]:
from itertools import product
def classes(D):
    qs = sorted(D["Q"])
    def lang(q, n):
        return frozenset(''.join(p) for k in range(n+1)
                         for p in product(sorted(D["Sigma"]), repeat=k)
                         if run_dfa_h(D, ''.join(p), q) in D["F"])
    n = len(qs)
    groups = {}
    for q in qs:
        groups.setdefault(lang(q, n), []).append(q)
    return sorted(map(sorted, groups.values()))

## 3. Tests

Three final states behave identically, so they form one class.

In [ ]:
cls = classes(D)
for c in cls: print("  class :", c)
print("\n%d states -> %d classes" % (len(D["Q"]), len(cls)))
assert any(len(c) >= 3 for c in cls), "F1, F2, F3 should coalesce"

Jove's `bash_eql_classes` performs the same coalescing on overlapping pairs.

In [ ]:
# bash_eql_classes takes a LIST OF PAIRS (tuples), not a list of sets,
# and returns [(representative, [members...]), ...] -- one entry per class.
demo = [('F1','F2'), ('F2','F3'), ('A','B')]
print("overlapping pairs :", demo)
for rep, members in bash_eql_classes(demo):
    print("   representative %-4s class %s" % (rep, sorted(members)))
print("\n(F1,F2) and (F2,F3) share F2, so they coalesce into one class of three.")
big = [sorted(m) for _, m in bash_eql_classes(demo) if len(m) >= 3]
assert big and set(big[0]) == {'F1','F2','F3'}

One class becomes one state, and the language is preserved.

In [ ]:
m = min_dfa(D)
print("minimal states :", sorted(m["Q"]))
print("count matches class count?", len(m["Q"]) == len(cls))
assert len(m["Q"]) == len(cls)
assert langeq_dfa(D, m)

The class containing $q_0$ is the new start state.

In [ ]:
print("original q0 : %s      minimal q0 : %s" % (D["q0"], m["q0"]))
print("minimal q0 name mentions the class members:", m["q0"])
assert accepts_dfa(m, '00') == accepts_dfa(D, '00')

## 4. Animation

The minimized machine &mdash; each state is a whole class of the original.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(D), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Why must indistinguishability be transitive for "bashing" to be well defined?
2. What goes wrong if you merge classes **before** pruning unreachable states?
3. Try `min_dfa(D, state_name_mode='verbose')`. What do the names show?

In [ ]:
# Your work for the exercises above.